# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [4]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [5]:
# TODO: Load environment variables
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [6]:
# retrieve_game tool
import chromadb
from chromadb.utils import embedding_functions

embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

# Connect to the persisted ChromaDB and load the existing collection
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)


@tool
def retrieve_game(query: str) -> list:
    """Semantic search: Finds the most relevant games in the vector DB.

    args:
    - query: a question about the game industry.

    You'll receive results as a list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(
        query_texts=[query],
        n_results=5,
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    games = []
    for doc, meta in zip(documents, metadatas):
        games.append({
            "Platform": meta.get("Platform"),
            "Name": meta.get("Name"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Description": meta.get("Description"),
            "Content": doc,
        })
    return games


#### Evaluate Retrieval Tool

In [ ]:
# evaluate_retrieval tool (LLM as judge)
from typing import List
from pydantic import BaseModel, Field
from lib.llm import LLM


class EvaluationReport(BaseModel):
    """Structured judgment about whether retrieved docs can answer the question."""
    useful: bool = Field(
        description="Whether the retrieved documents are enough to answer the question"
    )
    description: str = Field(
        description="Detailed explanation justifying the decision"
    )


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[dict]) -> dict:
    """Based on the user's question and on the list of retrieved documents,
    it analyzes the usability of the documents to respond to that question.

    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    llm = LLM(model="gpt-4o-mini", temperature=0.0)

    system_prompt = (
        "You are an evaluator of a RAG system for the video game industry. "
        "Your task is to evaluate if the documents are enough to respond to the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not."
    )
    user_prompt = (
        f"Question: {question}\n\n"
        f"Retrieved documents:\n{retrieved_docs}"
    )

    ai_message = llm.invoke(
        [SystemMessage(content=system_prompt), UserMessage(content=user_prompt)],
        response_format=EvaluationReport,
    )

    report = EvaluationReport.model_validate_json(ai_message.content)
    print(f"Evaluation result: {report}")
    return report.model_dump()


#### Game Web Search Tool

In [9]:
# game_web_search tool (Tavily)
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


@tool
def game_web_search(question: str) -> dict:
    """Web search: Looks up information about the game industry on the web.
    Use this when the internal vector DB does not contain enough information
    to answer the question.

    args:
    - question: a question about the game industry.
    """
    response = tavily_client.search(
        query=question,
        max_results=5,
        include_answer=True,
    )
    return response


### Agent

In [10]:
# Create the UdaPlay agent
instructions = (
    "You are UdaPlay, an AI research assistant for the video game industry. "
    "You answer questions about video games such as release years, platforms, "
    "genres and publishers.\n\n"
    "Follow this strategy for every question:\n"
    "1. Call `retrieve_game` to search the internal vector database first.\n"
    "2. Call `evaluate_retrieval` with the user's question and the retrieved "
    "documents to decide whether they are enough to answer.\n"
    "3. If the retrieval is NOT useful, call `game_web_search` to look the answer "
    "up on the web.\n"
    "4. Give a clear, concise final answer. Include the platform and release year "
    "when relevant, and mention whether the answer came from internal knowledge "
    "or from a web search.\n\n"
    "Never invent facts. If you cannot find the answer, say so."
)

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)


In [11]:
# Invoke the agent
questions = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for i, question in enumerate(questions):
    # Use a separate session per question so they don't share conversation history
    run = agent.invoke(question, session_id=f"q-{i}")
    final_state = run.get_final_state()
    answer = final_state["messages"][-1].content

    print(f"Q: {question}")
    print(f"A: {answer}")
    print("-" * 80)


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q: When was Pokémon Gold and Silver released?
A: Pokémon Gold and Silver were released in 1999 for the Game Boy Color. This information was retrieved from internal knowledge.
--------------------------------------------------------------------------------
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[S

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes